# L02 · 声音的数字表示——采样（sampling）、数组、第一个可听正弦波（sine wave）

**学习目标**
1. 理解采样：连续声音 → numpy 数组 → 可再生的声音
2. 实现 `samples_count`：N = duration × sample_rate
3. 实现 `make_time_axis`：生成时间轴
4. 理解数组形状（shape）：1D 信号 vs 2D 矩阵的区别
5. 实现 `make_sine`：生成正弦波并播放
6. 实现 `signal_summary`：汇总信号统计量

> 这四个函数是整个课程的「原子操作」——后续所有 notebook 都直接或间接调用它们。

← **上一课**　[L01 · Aurora 是什么](L01_motivation.ipynb)

> 上节课学习了 **Aurora 是什么**：从正弦波到 Whisper，6 个月路线图与核心动机。  
> 本课将探讨 **声音的数字表示**。

In [ ]:
# 统一导入（运行整个 notebook 前请先执行此单元）
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import Audio, display
except ImportError:
    pass


## 本课学习方法：先手算，再让代码验证

Aurora 课程的核心方法不是「运行看输出」，而是：

```
① 拿到公式，先在纸上或 Markdown 里推导
② 用极小的例子（例如 duration=0.5s, sr=8Hz → N=4）手算结果
③ 写代码实现
④ 运行 assert，让代码和手算结果对答案
```

**为什么这样做？**  
面试中没有终端，只有白板。「先手算」的练习
直接训练你在无代码环境中推导结果的能力。

本课每个任务都会先给一张推导表，等你填完再写代码。

## 1. 声音的物理本质

你有没有注意过，拨一根吉他弦时，空气会颤抖？

声音，本质上是空气**压强的波动**。吉他弦振动→推开附近空气分子→压强升高→传到你的耳膜→你听到了声音。麦克风做的事情完全一样：把空气压强的起伏，转成电压的起伏。

但"起伏"不能直接存进计算机。计算机只懂数字。

所以我们发明了**采样**（sampling）：每隔固定时间，量一次电压值，把它写成数字。这个"每隔多久量一次"就是**采样率**（sample rate，sr），单位是次/秒（Hz）。

CD 音质用 44100 Hz——每秒量 44100 次。人耳能听到的最高频率约 20000 Hz，根据奈奎斯特定理，采样率只需达到最高频率的 2 倍就能完整还原。44100 > 40000，刚好够用，还留了余量。

```
采样率 sr = 44100 Hz
时长 duration = 1 秒
采样点数 N = round(duration × sr) = 44100 个数字
```

一秒音频 = 44100 个浮点数。这就是"数字声音"的本质。

## ✏️ 任务 0：先手算 N，再实现 `samples_count`

**第一步：手算下表**（不要运行代码，用 `N = round(duration × sr)` 计算）

| duration (s) | sample_rate (Hz) | N（手算）|
|---|---|---|
| 0.5 | 8 | ✏️ ? |
| 1.0 | 16000 | ✏️ ? |
| 0.25 | 44100 | ✏️ ? |

**第二步**：填完表后再往下实现函数。

In [ ]:
def samples_count(duration, sample_rate):
    """返回给定时长和采样率下的采样点总数 N。"""
    # ✏️ TODO: 返回 round(duration * sample_rate)
    raise NotImplementedError("TODO: 返回 round(duration * sample_rate)")

In [ ]:
# 用代码对答案——和手算结果一致吗？
assert samples_count(0.5, 8)     == 4,     f'期望 4，得到 {samples_count(0.5, 8)}'
assert samples_count(1.0, 16000) == 16000, f'期望 16000，得到 {samples_count(1.0, 16000)}'
assert samples_count(0.25, 44100)== round(0.25 * 44100)
assert samples_count(1.0, 10)    == 10

# 打印对照表
print(f'  duration=0.5s,  sr=8     → N={samples_count(0.5, 8)}')
print(f'  duration=1.0s,  sr=16000 → N={samples_count(1.0, 16000)}')
print(f'  duration=0.25s, sr=44100 → N={samples_count(0.25, 44100)}')
print('samples_count  OK')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 用 8 Hz 采样率采 0.5 秒：只有 4 个点，可以逐个检查
sr, duration = 8, 0.5
N = samples_count(duration, sr)
t = np.arange(N) / sr
x = np.sin(2 * np.pi * 1.0 * t)

print(f'N = {N} 个点（{duration}s × {sr}Hz）')
print(f't = {t}  （秒）')
print(f'x = {np.round(x, 3)}  （采样值）')

plt.stem(t, x, markerfmt='C0o', linefmt='C0-', basefmt='k-')
plt.xlabel('时间 (s)'); plt.ylabel('振幅')
plt.title(f'8 Hz 采样率下的 1 Hz 正弦波（N={N} 点）')
plt.tight_layout(); plt.show()

## ✏️ 任务 1：实现 `make_time_axis(duration, sample_rate)`

**第一步：手算**（duration=0.5, sr=8）

| 步骤 | 公式 | 结果 |
|---|---|---|
| 1. 算 N | `samples_count(0.5, 8)` | ✏️ ? |
| 2. 生成编号 n | `np.arange(N)` | ✏️ [?, ?, ?, ?] |
| 3. 转为时间 t | `n / sample_rate` | ✏️ [?, ?, ?, ?] |

**推理路线**（写完手算再看这里）：
1. N = `samples_count(duration, sample_rate)`
2. 编号 n = `np.arange(N)` → `[0, 1, 2, ..., N-1]`
3. 时间轴 t = n / sample_rate

返回：dtype=float64 的 numpy 数组，第一个元素为 0，最后一个 < duration。

In [ ]:
def make_time_axis(duration, sample_rate):
    # ✏️ TODO: 使用 samples_count + np.arange 实现
    raise NotImplementedError("TODO: 使用 samples_count + np.arange 实现")

In [ ]:
import numpy as np

t = make_time_axis(0.5, 8)
assert isinstance(t, np.ndarray),          't 必须是 numpy 数组'
assert t.shape == (4,),                    f'shape 应为 (4,)，得到 {t.shape}'
assert np.allclose(t, [0.0, 0.125, 0.25, 0.375]), f'值不对：{t}'

t2 = make_time_axis(1.0, 16000)
assert t2.shape == (16000,)
assert abs(t2[0]) < 1e-12,   '第一个时刻应为 0'
assert t2[-1] < 1.0,         '最后一个时刻应 < duration（不含终点）'

print('make_time_axis  OK')
print(f'  t[:4] = {t}')

## 插曲：数组形状（shape）

想象你订了一张电影票——你需要知道两件事：**排号**和**座位号**。
NumPy 数组也一样。`shape` 就是"座位票根"，告诉你这个数组有几排、每排几个。

| 比喻 | NumPy | 含义 |
|---|---|---|
| 一列数字（一排座位） | shape `(N,)` | 1维，N 个元素 |
| 矩阵（多排座位） | shape `(M, N)` | 2维，M 行 N 列 |
| 标量（VIP 票，不占排） | shape `()` | 0维，就是一个数 |

音频信号通常是 `shape (N,)`——就是一排 N 个采样值的序列。
后续 STFT 输出会是 `(n_frames, n_fft//2+1)`——变成了矩阵，时间在行，频率在列。

搞清楚 shape，就不会在矩阵乘法时迷路。

In [ ]:
import numpy as np

scalar = np.float64(3.14)
vec    = np.array([0., 0.5, 1.])
mat    = np.array([[1, 2], [3, 4]])

for name, arr in [('标量', scalar), ('向量', vec), ('矩阵', mat)]:
    print(f'  {name:4s}  shape={str(arr.shape):10s}  ndim={arr.ndim}  dtype={arr.dtype}')

# 音频信号的形状
t = make_time_axis(1.0, 16000)
print(f'\n  音频时间轴  shape={t.shape}  → 一维，{len(t)} 个点')

## ✏️ 任务 2：实现 `make_sine(duration, sample_rate, frequency, amplitude=1.0)`

**第一步：手算**（duration=0.5s, sr=8Hz, freq=1Hz, A=1）

| n | t = n/sr (s) | θ = 2π·f·t (rad) | sin(θ)（精确到小数点后 2 位）|
|---|---|---|---|
| 0 | 0.000 | 0.000 | ✏️ ? |
| 1 | 0.125 | 0.785 | ✏️ ? |
| 2 | 0.250 | 1.571 | ✏️ ? |
| 3 | 0.375 | 2.356 | ✏️ ? |

**公式**：`x(t) = amplitude × sin(2π × frequency × t)`

**推理路线**：
1. 用 `make_time_axis` 生成时间轴 t
2. 返回 `amplitude × np.sin(2 × np.pi × frequency × t)`

In [ ]:
def make_sine(duration, sample_rate, frequency, amplitude=1.0):
    # ✏️ TODO: 使用 make_time_axis 和 np.sin 实现
    raise NotImplementedError("TODO: 使用 make_time_axis 和 np.sin 实现")

In [ ]:
import numpy as np

# 基本验证
x = make_sine(1.0, 16, 2.0)
assert isinstance(x, np.ndarray)
assert x.shape == (16,)
assert abs(x.max() - 1.0) < 1e-9,   f'max 应为 1.0，得到 {x.max()}'
assert abs(x.min() + 1.0) < 1e-9,   f'min 应为 -1.0，得到 {x.min()}'

# 振幅验证
x2 = make_sine(1.0, 16, 2.0, amplitude=3.0)
assert abs(x2.max() - 3.0) < 1e-9,  f'amplitude=3 时 max 应为 3.0'

# 与 numpy 参考一致
t = make_time_axis(1.0, 16)
ref = np.sin(2 * np.pi * 2.0 * t)
assert np.allclose(x, ref), '结果与 np.sin 参考不一致'

# 对照手算：duration=0.5, sr=8, freq=1
x_check = make_sine(0.5, 8, 1.0)
print('手算对照（freq=1Hz, sr=8Hz, duration=0.5s）：')
print(f'  代码输出 = {np.round(x_check, 3)}')
print(f'  参考值   = {np.round(np.sin([0, np.pi/4, np.pi/2, 3*np.pi/4]), 3)}')
print('make_sine  OK')

In [ ]:
import matplotlib.pyplot as plt

sr, dur = 200, 1.0
t = make_time_axis(dur, sr)

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, freq, color in zip(axes, [2, 5, 10], ['C0', 'C1', 'C2']):
    ax.plot(t, make_sine(dur, sr, freq), color=color)
    ax.set_ylabel(f'{freq} Hz'); ax.set_ylim(-1.3, 1.3); ax.grid(alpha=0.3)
axes[-1].set_xlabel('时间 (s)')
fig.suptitle('频率越高，单位时间内波峰越多')
plt.tight_layout(); plt.show()

try:
    from IPython.display import Audio, display
    x_audio = make_sine(1.5, 16000, 440)
    print('播放 440 Hz（A4 音）——你将听到标准「啦」音')
    display(Audio(x_audio, rate=16000))
except Exception:
    print('（非 Jupyter 环境，跳过播放）')

## 参数实验：频率与音高（pitch）

**预测问题（先回答，再运行验证）**：
- `make_sine(1.0, 16000, 880)` 的最大值是多少？
- 和 `make_sine(1.0, 16000, 440)` 的 shape 一样吗？
- 哪个频率听起来音调更高？

音高由频率决定。频率加倍 = 升高一个八度：

| 音名 | 频率 |
|---|---|
| A3（低八度）| 220 Hz |
| A4（标准「啦」）| 440 Hz |
| A5（高八度）| 880 Hz |

In [ ]:
try:
    from IPython.display import Audio, display
    sr = 16000
    for name, freq in [('A3 (220 Hz)', 220), ('A4 (440 Hz)', 440), ('A5 (880 Hz)', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')
        display(Audio(wave, rate=sr))
except Exception:
    sr = 16000
    for name, freq in [('A3', 220), ('A4', 440), ('A5', 880)]:
        wave = make_sine(1.0, sr, freq)
        print(f'{name}  shape={wave.shape}  max={wave.max():.3f}')

## ✏️ 任务 3：实现 `signal_summary(x)`

返回字典，包含信号的统计摘要：
- `length`：数组长度（整数）
- `shape`：数组形状（元组）
- `max_abs`：最大绝对值（浮点数）
- `mean`：均值（浮点数）
- `rms`：均方根（Root Mean Square）= √(mean(x²))（浮点数）

**手算**（x = make_sine(1.0, 4, 1.0)，即 4 个点：[0, 1, 0, -1]）：

| 键 | 公式 | 手算 |
|---|---|---|
| length | `len(x)` | ✏️ ? |
| shape | `x.shape` | ✏️ ? |
| max_abs | `max(|0|,|1|,|0|,|-1|)` | ✏️ ? |
| mean | `(0+1+0-1)/4` | ✏️ ? |
| rms | `√((0²+1²+0²+1²)/4)` | ✏️ ?（提示：纯正弦波 RMS = A/√2）|

In [ ]:
def signal_summary(x):
    # ✏️ TODO: 返回包含以下键的字典
    #   length  : len(x)
    #   shape   : x.shape
    #   max_abs : np.max(np.abs(x))
    #   mean    : float(np.mean(x))
    #   rms     : float(np.sqrt(np.mean(x**2)))  ← 均方根，白板挑战 Q4 会用到
    raise NotImplementedError("TODO: 返回含 length/shape/max_abs/mean/rms 的字典")

In [ ]:
import numpy as np

x_test = make_sine(1.0, 64, 3.0)
s = signal_summary(x_test)

assert s.get('length') == 64,                    f"length 应为 64，得到 {s.get('length')}"
assert s.get('shape') == (64,),                  f"shape 应为 (64,)，得到 {s.get('shape')}"
assert abs(s.get('max_abs', 0) - 1.0) < 0.02,   f"max_abs 应约为 1.0"
assert abs(s.get('mean', 1)) < 0.05,             f"均值应接近 0"
assert abs(s.get('rms', 0) - 1/np.sqrt(2)) < 0.02, f"rms 应约为 1/√2 ≈ 0.707"

# 手算验证：[0, 1, 0, -1] 的 4 个点
x4 = make_sine(1.0, 4, 1.0)
s4 = signal_summary(x4)
print('手算对照（4 个点 [0, 1, 0, -1]）：')
print(f'  x4     = {np.round(x4, 3)}')
print(f'  summary = {s4}')
print()
print('signal_summary  OK')
print(f'  rms 实测={s["rms"]:.4f}  理论值 1/√2={1/np.sqrt(2):.4f}')

## NumPy 三要素：dtype / 广播 / 切片

L32–L51 的 DSP 代码大量用到三个 NumPy 机制。这里用声音信号演示，为后续做铺垫。


In [ ]:
# ── dtype：float64 vs float32 ──────────────────────────────────────
# 音频处理两种常见精度
a64 = np.zeros(8, dtype=np.float64)
a32 = np.zeros(8, dtype=np.float32)
print(f'float64 每元素 {a64.itemsize} 字节，float32 每元素 {a32.itemsize} 字节')
print(f'float64 占用 {a64.nbytes} 字节  →  float32 占用 {a32.nbytes} 字节（省一半内存）')

# make_sine 默认生成 float64；转换到 float32：
t = make_time_axis(0.5, 8)
x64 = np.sin(2 * np.pi * 1.0 * t)           # float64，默认
x32 = x64.astype(np.float32)                  # float32，深度学习常用
print(f'\nmake_time_axis → dtype={x64.dtype},  转 float32 后 dtype={x32.dtype}')
print(f'精度差：max |x64 - x32| = {np.abs(x64 - x32.astype(np.float64)).max():.2e}')
# 为何音频通常用 float32？内存减半，GPU/神经网络原生支持；精度对人耳足够。


In [ ]:
# ── 广播（broadcasting）：叠加两路正弦波 ────────────────────────
# 标量乘数组：NumPy 自动将标量广播到每个元素
t = make_time_axis(1.0, 16)
wave_A = 0.7 * np.sin(2 * np.pi * 1.0 * t)  # 0.7 是标量，广播到 16 个元素
wave_B = 0.3 * np.sin(2 * np.pi * 3.0 * t)

# 两路信号相加：shape (16,) + shape (16,) → shape (16,)
mixed = wave_A + wave_B
print(f'wave_A.shape={wave_A.shape}, wave_B.shape={wave_B.shape} → mixed.shape={mixed.shape}')

# 多路信号堆叠为矩阵（行=通道，列=时间）
multi = np.stack([wave_A, wave_B, mixed])      # shape (3, 16)
print(f'堆叠后 multi.shape={multi.shape}  （3 个通道 × 16 个采样点）')
assert mixed.shape == (16,)
assert multi.shape == (3, 16)
print('✓ 广播与堆叠形状正确')


In [ ]:
# ── 切片（slicing）：截取与降采样 ───────────────────────────────
sr_demo, dur_demo = 8, 2.0                     # 8 Hz，2 秒，共 16 点
x = make_sine(dur_demo, sr_demo, 1.0)
print(f'完整信号 x.shape={x.shape}  →  {x.round(2)}')

# 切片 1：截取第 1 秒（索引 sr : 2*sr）
x_sec1 = x[sr_demo : 2 * sr_demo]
print(f'第 1 秒  x[{sr_demo}:{2*sr_demo}].shape={x_sec1.shape}  →  {x_sec1.round(2)}')

# 切片 2：降采样（每隔一点取一点 → 采样率减半）
x_down = x[::2]
print(f'降采样  x[::2].shape={x_down.shape}  →  {x_down.round(2)}')

assert x_sec1.shape == (sr_demo,), f'期望 ({sr_demo},)，得到 {x_sec1.shape}'
assert x_down.shape == (sr_demo,), f'期望 ({sr_demo},)，得到 {x_down.shape}'
print('✓ 切片形状正确')


## Aurora 连接

本课实现的四个函数在 `src/aurora/` 中均有对应实体：

| 本课函数 | Aurora 模块 | 说明 |
|---|---|---|
| `samples_count` | `src/aurora/audio/transforms.py` | 采样点数计算，所有信号函数的第一步 |
| `make_time_axis` | `src/aurora/audio/transforms.py` | 时间轴生成工具 |
| `make_sine` | `src/aurora/audio/transforms.py` | 测试桩 & L32 信号生成器 |
| `signal_summary` | `src/aurora/audio/transforms.py` | 信号统计摘要 |

安装包后可直接导入：

```python
from aurora.audio.transforms import make_sine, make_time_axis
```

L32（`notebooks/5_audio_dsp/L32_numpy_signals.ipynb`）将直接复用这些函数构建完整的信号处理管线。


## ✏️ 白板挑战：不看代码，纸上推导（目标 10 分钟）

这是你学完本课后的闭卷自测。**盖上屏幕**，在纸上完成以下推导：

---

**题目**：信号参数如下
- 采样率 `sr = 8000` Hz（电话音质）
- 时长 `duration = 0.5` 秒
- 频率 `freq = 440` Hz（A4 音）
- 振幅 `amplitude = 0.7`

**问 1**：数组 `x` 有多少个元素？（写出公式）

**问 2**：`t[100]` 等于多少秒？（`t` 是时间轴数组）

**问 3**：`x[100]` 的值是多少？（用 `sin` 表示，不用算出精确值）

**问 4**：`signal_summary(x)['rms']` 的理论值是多少？
提示：纯正弦波 `A·sin(...)` 的 RMS = A / √2

---

推导完成后运行下面的验证格对答案。

In [ ]:
# ✏️ 对答案格——先完成白板推导再运行
import numpy as np

sr, duration, freq, amplitude = 8000, 0.5, 440, 0.7

# 问1：元素个数
N_expected = round(duration * sr)
t = make_time_axis(duration, sr)
x = make_sine(duration, sr, freq, amplitude)
assert len(x) == N_expected, f"Q1: 期望 {N_expected}，实际 {len(x)}"
print(f"Q1 ✅  N = round({duration} × {sr}) = {N_expected} 个采样点")

# 问2：t[100]
t100_expected = 100 / sr
assert np.isclose(t[100], t100_expected, atol=1e-10), f"Q2: 期望 {t100_expected:.6f}，实际 {t[100]:.6f}"
print(f"Q2 ✅  t[100] = 100 / {sr} = {t100_expected:.6f} 秒")

# 问3：x[100]
x100_expected = amplitude * np.sin(2 * np.pi * freq * t[100])
assert np.isclose(x[100], x100_expected, atol=1e-10)
print(f"Q3 ✅  x[100] = {amplitude} × sin(2π × {freq} × {t100_expected:.6f}) = {x100_expected:.6f}")

# 问4：RMS 理论值
rms_theory = amplitude / np.sqrt(2)
s = signal_summary(x)
assert np.isclose(s['rms'], rms_theory, atol=0.01), f"Q4: 理论 {rms_theory:.4f}，实际 {s['rms']:.4f}"
print(f"Q4 ✅  RMS ≈ {amplitude}/√2 = {rms_theory:.4f}  实测 {s['rms']:.4f}")
print()
print("🎉 白板挑战通过！你对采样、时间轴、正弦波的计算已经内化。")

In [ ]:
# ✏️ 本课自评
l02_review = {
    "samples_count_done":   None,  # samples_count 实现并通过断言？True/False
    "make_time_axis_done":  None,  # make_time_axis 实现并通过断言？True/False
    "make_sine_done":       None,  # make_sine 实现并通过断言？True/False
    "signal_summary_done":  None,  # signal_summary 实现并通过断言？True/False
    "whiteboard_passed":    None,  # 白板挑战（纸上推导）完成？True/False
    "sampling_rate_intuition": None,  # 能解释为什么 sr=44100 而不是 44000？True/False
}

unfilled = [k for k, v in l02_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l02_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
    print('   建议：重新阅读对应章节，用手算而不是运行代码来验证理解。')
else:
    print('✅ L02 全部通关！进入 L03：谱图直觉')

## 本课收束

| 函数 | 作用 | 后续出现 |
|---|---|---|
| `samples_count` | N = duration × sr | 每个信号函数的第一步 |
| `make_time_axis` | 生成时间轴 | L32 numpy_signals, L33 sine_wave |
| `make_sine` | 生成正弦波 | L34 aliasing, L36 windows, L37 DFT |
| `signal_summary` | 汇总统计量 | L40 spectrum, L51 real_audio |

**本课用到的方法论**：先手算推导表，再写代码对答案。
这个方法贯穿整个 Aurora 课程——FFT 的每一步也会先手算再实现。

**核心直觉**：声音 = 一维 float64 数组，长度 = duration × sample_rate。
FFT 把这个数组变换到频域（frequency domain）——那是 L37 以后的事。

**下一课 L03**：谱图直觉——在学 FFT 之前先看结果。
你将看到上面各类声音的时频图，为 L37-L41 种下视觉直觉。

---

→ **下一课**　[L03 · 谱图直觉](L03_spectrogram.ipynb)

> 下节课将学习 **谱图直觉**：在学 FFT 之前先读懂时频图的三个轴。